# 05c | Drills: probability

Covers **05a and 05b**: Bayes, `scipy.stats` for common distributions, verifying answers by simulation, and several classic verbal questions in new wording.

**How to use this lesson:** there are no input boxes here. There are two kinds of problems:

- **Calculation problems**: the code and its output are given directly below. First think about which function and formula to use, then read on;
- **Verbal problems** (marked 🗣 after the title): answer out loud first (30–60 seconds), then compare with the reference answer.

Problems on the same page run in order, so variables and functions defined earlier can be reused later. The following has already been run at the top of every problem:

```python
import numpy as np
from scipy import stats
```

### Problem 1 · Bayes as a function

A disease has a prevalence of 2%, and a test has 90% sensitivity and 90% specificity. After a positive test, what is the probability of actually having the disease?

```python
def posterior(prior, sens, spec):
    true_pos = sens * prior                  # sick and positive
    false_pos = (1 - spec) * (1 - prior)     # healthy but positive
    return true_pos / (true_pos + false_pos)

round(posterior(0.02, 0.90, 0.90), 4)
```

**Result:**

```text
0.1552
```

> Only about 15.5%. With low prevalence, false positives far outnumber true positives.

### Problem 2 · Testing again

The same person tests positive a second time (assume the two tests are independent given disease status). What is the probability of disease now?

```python
p1 = posterior(0.02, 0.90, 0.90)
p2 = posterior(p1, 0.90, 0.90)       # the first posterior becomes the second prior
print(round(p1, 4), round(p2, 4))
```

**Result:**

```text
0.1552 0.6231
```

> Approach: use the posterior from the first test as the prior for the second, and apply Bayes again. After two positives the probability rises to about 62%.

### Problem 3 · Binomial: at least k

You guess on all 20 four-option multiple-choice questions. What is the probability of getting at least 10 right?

```python
stats.binom.sf(9, 20, 0.25)
```

**Result:**

```text
0.0139
```

> `sf(9)` is P(X > 9) = P(X ≥ 10). "At least k" is `sf(k - 1)`; this is the easiest place to make an off-by-one error.

### Problem 4 · Poisson: more than k

You receive 4 emails per hour on average. What is the probability of receiving **more than** 6 in a given hour?

```python
stats.poisson.sf(6, 4)
```

**Result:**

```text
0.1107
```

> "More than 6" is P(X > 6) = `sf(6)`.

### Problem 5 · The exponential distribution and memorylessness

A bus comes every 10 minutes on average (arrivals form a Poisson process). ① The probability of waiting more than 15 minutes; ② having already waited 10 minutes, the probability of waiting more than 15 more.

```python
wait = stats.expon(scale=10)                  # scale is the mean 1/λ
p_15 = wait.sf(15)
p_more_15 = wait.sf(25) / wait.sf(10)         # P(T > 25 | T > 10)
print(round(p_15, 4), round(p_more_15, 4))
```

**Result:**

```text
0.2231 0.2231
```

> The two numbers are the same, both $e^{-1.5}$: how long you've already waited doesn't affect how much longer you'll wait. Note that scipy parameterizes the exponential by `scale` (the mean), not the rate.

### Problem 6 · Normal: quantiles and interval probabilities

Height ~ N(170, 8²). ① Where does the tallest 5% start, in cm? ② What proportion is between 160 and 180?

```python
cut = stats.norm.ppf(0.95, 170, 8)
between = stats.norm.cdf(180, 170, 8) - stats.norm.cdf(160, 170, 8)
print(round(cut, 2), round(between, 4))
```

**Result:**

```text
183.16 0.7887
```

> `ppf` is the quantile function (R's `qnorm`), `cdf` is P(X ≤ x) (R's `pnorm`). Interval probability = the difference of two cdfs.

### Problem 7 · Geometric distribution

Each shot goes in with probability 0.2. ① The probability that the first make is on the 3rd shot; ② the probability of a make within 5 shots; ③ the expected shot number of the first make.

```python
print(round(stats.geom.pmf(3, 0.2), 4))     # 0.8 * 0.8 * 0.2
print(round(stats.geom.cdf(5, 0.2), 4))     # 1 - 0.8 ** 5
print(1 / 0.2)
```

**Result:**

```text
0.128
0.6723
5.0
```

### Problem 8 · Verify by simulation: two dice

The probability that two dice sum to at least 10. Work out the theoretical value first, then verify by simulation.

```python
rng = np.random.default_rng(0)
n = 200_000
d1 = rng.integers(1, 7, n)        # 1 to 6
d2 = rng.integers(1, 7, n)
print(round(((d1 + d2) >= 10).mean(), 4), round(6 / 36, 4))
```

**Result:**

```text
0.1665 0.1667
```

> Sums of 10, 11, and 12 have 3, 2, and 1 combinations respectively, 6/36 in total. The standard simulation recipe: generate many trials at once → check the event → take the mean.

### Problem 9 · The birthday problem: 30 people

The probability that at least two of 30 people share a birthday.

```python
def birthday(n):
    p_all_different = 1.0
    for i in range(n):
        p_all_different *= (365 - i) / 365
    return 1 - p_all_different

round(birthday(30), 4)
```

**Result:**

```text
0.7063
```

> Compute "all different" first, then subtract from 1. It already exceeds 50% at 23 people.

### Problem 10 · Coupon collector: 10 card types

There are 10 equally likely card types. How many draws does it take on average to collect them all?

```python
def coupon_expected(n):
    return sum(n / (n - k) for k in range(n))

round(coupon_expected(10), 2)
```

**Result:**

```text
29.29
```

> With k types already collected, the chance of drawing a new one is (n−k)/n, so you expect n/(n−k) draws; add up the stages (linearity of expectation).

### Problem 11 · Simulation: waiting for "heads, tails"

Keep flipping a coin until **two consecutive flips** come up "heads, then tails". How many flips do you need on average? Estimate it by simulation.

```python
def flips_until_ht(rng):
    count = 0
    prev_head = False
    done = False
    while not done:
        count += 1
        head = rng.random() < 0.5
        if prev_head and not head:
            done = True
        prev_head = head
    return count

rng = np.random.default_rng(1)
round(np.mean([flips_until_ht(rng) for _ in range(20000)]), 3)
```

**Result:**

```text
4.008
```

> The theoretical value is 4, less than the 6 for "heads, heads". Why: when waiting for "heads, heads", a tail on the second flip sends you back to the start; when waiting for "heads, tails", once you've flipped a head, any number of further heads still leaves you in the "already have one head" state.

### Problem 12 · CLT: the standard deviation of the sample mean

Draw 30 numbers from Uniform(0, 1) and average them; repeat 10,000 times. What is the standard deviation of these averages? Compare it with the theoretical value.

```python
rng = np.random.default_rng(2)
means = rng.random(size=(10000, 30)).mean(axis=1)     # each row is one sample
print(round(means.std(), 4), round(np.sqrt(1 / 12 / 30), 4))
```

**Result:**

```text
0.053 0.0527
```

> The variance of Uniform(0, 1) is 1/12, and the standard error of the mean is $\sigma/\sqrt n$. The CLT says the distribution of this **mean** is approximately normal, not the raw data.

### Problem 13 · The two-child problem 🗣

A family has two children, and at least one of them is a boy. What is the probability that both are boys?

**Reference answer:**

1/3. The equally likely cases are boy-boy, boy-girl, girl-boy, girl-girl; "at least one boy" rules out girl-girl, and only 1 of the remaining 3 cases is two boys.

Common trap: if the question instead says "the older child is a boy", the answer becomes 1/2. Different conditions, different sample spaces.

*Sample phrasing:* One third. Conditioning on at least one boy leaves three equally likely cases, and only one is two boys. If we knew the older child is a boy, it would be one half.

### Problem 14 · Linearity of expectation: the hat-check problem 🗣

10 people put their hats in a pile, then each takes one at random. On average, how many people get their own hat back?

**Reference answer:**

1 on average. Let the indicator $I_k$ = person k gets their own hat, with $P(I_k = 1) = 1/10$. The total is $\sum_k I_k$, so the expectation is $10 \times 1/10 = 1$.

The $I_k$ are not independent, but **linearity of expectation does not require independence**, which is exactly why the indicator method is so handy. The answer doesn't depend on the number of people: it's 1 for n people too.

*Sample phrasing:* One. Write the count as a sum of indicators, each with probability 1/n; linearity of expectation doesn't need independence.

### Problem 15 · Var(X − Y) 🗣

X and Y both have variance 4. ① What is Var(X − Y) if X and Y are independent? ② What if their correlation is 0.5?

**Reference answer:**

$\mathrm{Var}(X - Y) = \mathrm{Var}(X) + \mathrm{Var}(Y) - 2\,\mathrm{Cov}(X, Y)$.

① With independence the covariance is 0, so the result is 4 + 4 = 8 (variances add, they don't subtract). ② $\mathrm{Cov} = 0.5 \times 2 \times 2 = 2$, so the result is 8 − 4 = 4.

This is why paired designs (measuring the same person before and after) are more precise: the two measurements are positively correlated, so the variance of the difference shrinks.

### Problem 16 · E[X²] 🗣

Given E[X] = 2 and Var(X) = 3, what is E[X²]?

**Reference answer:**

$E[X^2] = \mathrm{Var}(X) + E[X]^2 = 3 + 4 = 7$.

This rearrangement ($\mathrm{Var}(X) = E[X^2] - E[X]^2$) is the most common step in expectation and variance problems, and it's often used in reverse too: getting the variance from the second moment.

### Problem 17 · A fair result from an unfair coin 🗣

All you have is a coin whose probability of heads is unknown (and not 0 or 1). How do you use it to make a 50/50 decision?

**Reference answer:**

Flip it twice: "heads, tails" means A, "tails, heads" means B, and "heads, heads" or "tails, tails" means discard and start over. Since $P(\text{HT}) = p(1-p) = P(\text{TH})$, A and B are equally likely.

The expected number of rounds is $1 / [2p(1-p)]$, which gets slower the further p is from 0.5. This is known as the von Neumann trick.

*Sample phrasing:* Flip twice; heads-tails means A, tails-heads means B, and repeat on a tie. Both outcomes have probability p times one minus p.

### Problem 18 · The maximum of two uniforms 🗣

X and Y are independent, both Uniform(0, 1). What is the expected value of max(X, Y)?

**Reference answer:**

2/3. $P(\max \le t) = P(X \le t)P(Y \le t) = t^2$, so the density is $2t$ and the expectation is $\int_0^1 t \cdot 2t\,dt = 2/3$.

Generalization: the expected maximum of n independent Uniform(0, 1) variables is $n/(n+1)$, and the expected minimum is $1/(n+1)$. "The CDF of the maximum = the product of the individual CDFs" is the standard starting point for problems like this.

### Problem 19 · The relationship between Poisson and binomial 🗣

When can you approximate a binomial with a Poisson? Give a business scenario.

**Reference answer:**

When the number of trials n is large, the success probability p of each is small, and $np$ is moderate, Binomial(n, p) ≈ Poisson(λ = np).

Scenario: a site gets 100,000 visits a day, and each visit has a three-in-one-hundred-thousand chance (0.003%) of an error; the number of errors per day is approximately Poisson(3). You then need only one parameter, and the math is convenient.

Another intuition: the Poisson describes how many of "a large number of independent, low-probability events" happen in a fixed time, and its mean equals its variance. If the real data's variance is much larger than its mean (overdispersion), the Poisson is not appropriate.

### Problem 20 · LLN and CLT: the casino example 🗣

Use a casino to explain what the law of large numbers and the central limit theorem each say.

**Reference answer:**

On each bet, the casino's expected profit is a small positive number (say 2.7 cents per dollar wagered).

**Law of large numbers:** the more bets there are, the closer the casino's average profit per bet gets to that 2.7 cents, so the casino wins reliably in the long run. It answers **where** the average lands.

**Central limit theorem:** after n bets, the fluctuation of the average profit around 2.7 cents is approximately normal, with a standard deviation that shrinks like $1/\sqrt n$. It answers **how it fluctuates around there**, and you can use it to compute "the probability of losing money on a given day".

*Sample phrasing:* The law of large numbers says the casino's average take converges to its edge; the CLT says the fluctuations around that edge are approximately normal and shrink like one over root n.

## Recap

| Goal | Code / key point |
|---|---|
| Bayes | True positives ÷ (true positives + false positives); for repeated tests, the posterior becomes the prior |
| P(X ≥ k) | `stats.binom.sf(k - 1, n, p)` |
| Quantile / interval probability | `stats.norm.ppf(q, mu, sd)` / the difference of two `cdf`s |
| Exponential distribution | `stats.expon(scale=mean)`; memorylessness |
| Geometric distribution | `stats.geom.pmf(k, p)`, mean 1/p |
| Simulation | Generate many trials → check the event → `.mean()` |
| Expectation problems | Indicators + linearity; coupon collector $\sum n/(n-k)$ |
| Variance | $\mathrm{Var}(X \pm Y) = \mathrm{Var}X + \mathrm{Var}Y \pm 2\mathrm{Cov}$, $E[X^2] = \mathrm{Var} + E^2$ |